# INTRODUCTION AND OUTLINE #

This project creates predictions for every match for the rest of the season for any of the top 5 European Leagues, based on optimised goal, xG and betting line ratings. Defining the league variable by one of the names listed above it allows you to choose which league you would like to look at.

These predictions are then used to create a league table with predicted number of points, goals scored and goals conceded at the end of the season.
It also contains predictions for each teams likelihood to finish 1st, Top 4, Top 6, Top 8 and Bottom 3, derived from running Monte Carlo simulations on the rest of the seasons fixtures.
NOTE: The data has been cut off to simulate the rest of the 2025/2026 season as if it is still 01/04/2026.


The project is split up into the following sections

1. Understat Fixture Scraper
2. Data Clean Up
3. TotalCorner Dataset Import and Clean Up
4. Dataset Merging
5. Home Advantage Factor
6. Team Performance Dataframe
7. Finding Optimal Parameters For Assessing Teams Performance
8. Defining Optimal Parameters and Testing
9. Applying Optimal Parameters to Future Fixtures
10. Assessing Draw Factor in Historical Results
11. Calculating a Draw Adjustment Factor
10. Current Season Table
11. End of Season Table Prediction
12. Monte Carlo Simulations for Rest of Season
13. Limitations and Further Improvements



### PACKAGE IMPORTS ###

In [1]:
import pandas as pd
import understatapi
import numpy as np
from itertools import product
from scipy.stats import poisson,skellam
from scipy.optimize import brentq
from sklearn.metrics import mean_absolute_error, mean_squared_error
from datetime import date
from scipy.optimize import minimize_scalar

### UNDERSTAT FIXTURE SCRAPER ###
This section scrapes fixture data from understat for a given league and combines into a single data frame

In [2]:
# League to import

#EPL
#La_Liga
#Serie_A
#Bundesliga
#Ligue_1

league = 'EPL'

# Seasons to import
seasons = range(2015, 2026)

# Date to simulate as 'today' - matches after this date are treated as not yet played, leaving fixtures for the model to predict
season_cutoff_date = date(2026, 4, 1)

# Defines league variables
league_data = {
    'EPL': {
        'league_tc': 'EnglandPremierLeague',
        'n_teams_div': 20
    },
    'La_Liga': {
        'league_tc': 'SpainLaLiga',
        'n_teams_div': 20
    },
        'Serie_A': {
        'league_tc': 'ItalySerieA',
        'n_teams_div': 20
    },
        'Bundesliga': {
        'league_tc': 'GermanyBundesligaI',
        'n_teams_div': 18
    },
        'Ligue_1': {
        'league_tc': 'FranceLigue1',
        'n_teams_div': 18
    }
}

# Defines additional variables for league
league_info = league_data[league]
league_tc = league_info['league_tc']
n_teams_div = league_info['n_teams_div']

# Sets up client
client = understatapi.UnderstatClient()

# Sets up list to store each season 
raw_data = []

# Loop to scrape data for each season, define 'season' column and append seasons to raw_data list
for season in seasons:
    data = client.league(league=league).get_match_data(season=str(season))
    df = pd.DataFrame(data)
    df['season'] = f"{season}/{season + 1}"
    raw_data.append(df)

# Combine all the data into a single DataFrame
fixtures_data = pd.concat(raw_data, ignore_index=True)

### DATA CLEANUP ###
This section reformats the data from a json format, drops unneccessary columns, converts the datetime column datatype and renames some columns

In [3]:
# Fixes formatting
dict_columns = ['h', 'a', 'goals', 'xG', 'forecast']

for col in dict_columns:
    formatted_df = pd.json_normalize(fixtures_data[col])
    formatted_df.columns = [f"{col}_{key}" for key in formatted_df.columns]
    fixtures_data = pd.concat([fixtures_data, formatted_df], axis=1)


# Removes redundant columns
fixtures_data.drop(['h', 'a', 'goals', 'xG', 'forecast', 'id', 'h_id', 
                    'h_short_title', 'a_id', 'a_short_title', 'forecast_w', 
                    'forecast_d', 'forecast_l'], axis = 1, inplace=True)

   
# Formats date column and renames columns
fixtures_data['datetime'] = pd.to_datetime(fixtures_data['datetime']).dt.strftime('%Y-%m-%d')
fixtures_data = fixtures_data.rename(columns  = {'h_title':'Home', 'a_title':'Away', 'datetime': 'Date'})

# Simulates 'today' as the cutoff date - marks matches after this point as not yet played, so the model treats them as fixtures to predict rather than results to learn from
fixtures_data.loc[pd.to_datetime(fixtures_data['Date']).dt.date > season_cutoff_date, 'isResult'] = False

In [4]:
fixtures_data.head(10)

,isResult,Date,season,Home,Away,goals_h,goals_a,xG_h,xG_a
0,True,2015-08-08,2015/2016,Manchester United,Tottenham,1,0,0.627539,0.6746
1,True,2015-08-08,2015/2016,Bournemouth,Aston Villa,0,1,0.876106,0.782253
2,True,2015-08-08,2015/2016,Everton,Watford,2,2,0.604226,0.557892
3,True,2015-08-08,2015/2016,Leicester,Sunderland,4,2,2.56803,1.45946
4,True,2015-08-08,2015/2016,Norwich,Crystal Palace,1,3,1.13076,2.10975
5,True,2015-08-08,2015/2016,Chelsea,Swansea,2,2,0.64396,2.59203
6,True,2015-08-09,2015/2016,Newcastle United,Southampton,2,2,1.54613,1.2529
7,True,2015-08-09,2015/2016,Arsenal,West Ham,0,2,1.33166,0.535961
8,True,2015-08-09,2015/2016,Stoke,Liverpool,0,1,0.381274,0.329873
9,True,2015-08-10,2015/2016,West Bromwich Albion,Manchester City,0,3,0.435238,1.9242


We now have a clean dataframe with all past and future fixtures, with goals and xG performances for each team

### TOTALCORNER DATASET IMPORT AND CLEAN UP ###
Here we are importing another dataset that contains match odds for all fixtures, we will merge this with our understat data to get odds for each fixture

In [5]:
# Imports totalcorner fixtures
fixtures_tc = pd.read_csv(f"data/{league_tc}.csv")

# Formats date column
fixtures_tc['Date'] = pd.to_datetime(fixtures_tc['Date'], format='%d.%m.%Y').dt.date

# Creates unique match identifier and reduces to one row per match
fixtures_tc['matchid'] = fixtures_tc['Date'].astype(str) +"-"+ fixtures_tc["Home"] +"-"+ fixtures_tc["Away"]
fixtures_tc = fixtures_tc.drop_duplicates(subset='matchid', keep='first')

# Converts AH Line and Goal Line strings into a single numeric value
def parse_line(value):
    if isinstance(value, str):
        value = value[6:-1]
    value = str(value)
    if ',' in value:
        a, b = (p.strip() for p in value.split(','))
        return (float(a) + float(b)) / 2
    return float(value)

fixtures_tc['hcaplevel'] = fixtures_tc['AH.Line'].apply(parse_line)
fixtures_tc['sup_level'] = 0 - fixtures_tc['hcaplevel']
fixtures_tc['goallevel'] = fixtures_tc['Goal.Line'].apply(parse_line)

# Removes margin from a pair of odds to get demargined probabilities
def remove_margin(odds_a, odds_b):
    p_a_raw = 1 / odds_a
    p_b_raw = 1 / odds_b
    total = p_a_raw + p_b_raw
    return p_a_raw / total, p_b_raw / total

# Probability that total match goals go over an asian totals line
def asian_over_probability(lam, line):
    remainder = line % 1.0

    if remainder == 0.0:
        return (1 - poisson.cdf(line, lam)) + 0.5 * poisson.pmf(line, lam)
    elif remainder == 0.5:
        return 1 - poisson.cdf(line - 0.5, lam)
    elif remainder == 0.25:
        lower = line - 0.25
        upper = line + 0.25
        p_lower = (1 - poisson.cdf(lower, lam)) + 0.5 * poisson.pmf(lower, lam)
        p_upper = 1 - poisson.cdf(upper - 0.5, lam)
        return 0.5 * (p_lower + p_upper)
    else:
        lower = line - 0.25
        upper = line + 0.25
        p_lower = 1 - poisson.cdf(lower - 0.5, lam)
        p_upper = (1 - poisson.cdf(upper, lam)) + 0.5 * poisson.pmf(upper, lam)
        return 0.5 * (p_lower + p_upper)

# Solves for the total goals rate implied by an asian totals line and its over/under odds
def fit_lambda(line, over_odds, under_odds):
    if pd.isna(line) or pd.isna(over_odds) or pd.isna(under_odds):
        return np.nan
    p_over, _ = remove_margin(over_odds, under_odds)
    objective = lambda lam: asian_over_probability(lam, line) - p_over
    try:
        return brentq(objective, 0.1, 15.0)
    except ValueError:
        return np.nan

fixtures_tc['asian_total_goals'] = fixtures_tc.apply(lambda row: fit_lambda(row['goallevel'], row['Goal.O.Odds'], row['Goal.U.Odds']), axis=1)

# Probability that the home team covers an asian handicap line
def skellam_home_prob(delta, line, base):
    mu_home = max(base + delta / 2, 0.01)
    mu_away = max(base - delta / 2, 0.01)

    remainder = line % 1.0
    if remainder < 0:
        remainder += 1.0

    if remainder == 0.0:
        p_win = 1 - skellam.cdf(line, mu_home, mu_away)
        p_push = skellam.pmf(int(line), mu_home, mu_away)
        return p_win + 0.5 * p_push
    elif remainder == 0.5:
        return 1 - skellam.cdf(line - 0.5, mu_home, mu_away)
    elif remainder == 0.25:
        lower = line - 0.25
        upper = line + 0.25
        p_lower = 1 - skellam.cdf(lower, mu_home, mu_away) + 0.5 * skellam.pmf(int(lower), mu_home, mu_away)
        p_upper = 1 - skellam.cdf(upper - 0.5, mu_home, mu_away)
        return 0.5 * (p_lower + p_upper)
    else:
        lower = line - 0.25
        upper = line + 0.25
        p_lower = 1 - skellam.cdf(lower - 0.5, mu_home, mu_away)
        p_upper = 1 - skellam.cdf(upper, mu_home, mu_away) + 0.5 * skellam.pmf(int(upper), mu_home, mu_away)
        return 0.5 * (p_lower + p_upper)

# Solves for the goal difference implied by an asian handicap line and its home/away odds
def fit_delta(line, home_odds, away_odds, base):
    if pd.isna(line) or pd.isna(home_odds) or pd.isna(away_odds) or pd.isna(base):
        return np.nan
    p_home, _ = remove_margin(home_odds, away_odds)
    objective = lambda delta: skellam_home_prob(delta, line, base) - p_home
    bound = max(2 * base - 0.01, 0.01)
    try:
        return brentq(objective, -bound, bound)
    except ValueError:
        return np.nan

fixtures_tc['asian_sup'] = fixtures_tc.apply(lambda row: fit_delta(row['sup_level'], row['AH.Home.Odds'], row['AH.Away.Odds'], row['asian_total_goals'] / 2), axis=1)

# Restricts the totalcorner dataset to matches before the cutoff, so lines for not-yet-played fixtures aren't included
fixtures_tc = fixtures_tc[fixtures_tc['Date'] < season_cutoff_date]

### DATASET MERGING ###

Now we have our two datasets in a clean and manageble format we can combine them. Here we identify the team names and fix any name discrepancies between the datasets. We then merge the datasets to add goals and supremacy totals derived from betting lines to the fixtures, goals and xG from understat. 

In [6]:
# Identifies teams to harmonise names
sorted_teams_us = sorted(fixtures_data['Home'].unique())
sorted_teams_tc = sorted(fixtures_tc['Home'].unique())

# Converts names
name_conv = {'Parma Calcio 1913':'Parma', 'Inter':'Inter Milan', 'SPAL 2013':'Spal',
             'Wolverhampton Wanderers':'Wolverhampton', 'West Bromwich Albion':'West Brom', 'Sheffield United':'Sheff Utd',
             'Manchester City':'Man City', 'Manchester United':'Man Utd', 'Newcastle United':'Newcastle', 'Nottingham Forest':'Nottm Forest',
             'Alaves':'CD Alaves', 'Athletic Club':'Athletic Bilbao', 'Real Valladolid': 'Valladolid', 'SD Huesca':'Huesca',
             'Borussia M.Gladbach':'Borussia M\'gladbach', 'FC Cologne':'Cologne', 'FC Heidenheim':'Heidenheim','Freiburg': 'SC Freiburg',
             'Hamburger SV':'Hamburg', 'Hoffenheim':'TSG Hoffenheim', 'Ingolstadt':'FC Ingolstadt', 'Mainz 05':'Mainz',
             'Nuernberg':'Nurnberg', 'RasenBallsport Leipzig':'RB Leipzig', 'Schalke 04':'Schalke', 'St. Pauli':'St Pauli',
             'Fortuna Duesseldorf':'Fortuna Dusseldorf', 'Ajaccio':'AC Ajaccio', 'GFC Ajaccio':'Ajaccio GFCA',
             'Paris Saint Germain':'PSG', 'Saint-Etienne':'St Etienne'
             }

fixtures_data['Home'] = fixtures_data['Home'].replace(name_conv)
fixtures_data['Away'] = fixtures_data['Away'].replace(name_conv)

# Creates unique match identifier
fixtures_data['matchid'] = fixtures_data['Date'].astype(str) +"-"+ fixtures_data["Home"] +"-"+ fixtures_data["Away"]

# Merges to attach odds data
fixtures_data = fixtures_data.merge(
    fixtures_tc[['matchid', 'asian_sup', 'asian_total_goals']],
    on=['matchid'],
    how='left'
)

# Adds missing data for rogue Bundesliga match
fixtures_data.loc[fixtures_data['matchid'] == '2025-02-09-Holstein Kiel-Bochum', ['isResult', 'goals_h', 'goals_a', 'xG_h', 'xG_a']] = [True, 2, 2, 1.37, 2.05]

In [7]:
fixtures_data.head(10)

,isResult,Date,season,Home,Away,goals_h,goals_a,xG_h,xG_a,matchid,asian_sup,asian_total_goals
0,True,2015-08-08,2015/2016,Man Utd,Tottenham,1,0,0.627539,0.6746,2015-08-08-Man Utd-Tottenham,0.865951,2.752845
1,True,2015-08-08,2015/2016,Bournemouth,Aston Villa,0,1,0.876106,0.782253,2015-08-08-Bournemouth-Aston Villa,0.687761,2.648149
2,True,2015-08-08,2015/2016,Everton,Watford,2,2,0.604226,0.557892,2015-08-08-Everton-Watford,0.704262,2.648149
3,True,2015-08-08,2015/2016,Leicester,Sunderland,4,2,2.56803,1.45946,2015-08-08-Leicester-Sunderland,0.736740,2.495550
4,True,2015-08-08,2015/2016,Norwich,Crystal Palace,1,3,1.13076,2.10975,2015-08-08-Norwich-Crystal Palace,0.140291,2.391947
5,True,2015-08-08,2015/2016,Chelsea,Swansea,2,2,0.64396,2.59203,2015-08-08-Chelsea-Swansea,1.412795,2.752845
6,True,2015-08-09,2015/2016,Newcastle,Southampton,2,2,1.54613,1.2529,2015-08-09-Newcastle-Southampton,-0.183505,2.417580
7,True,2015-08-09,2015/2016,Arsenal,West Ham,0,2,1.33166,0.535961,2015-08-09-Arsenal-West Ham,1.889021,3.248678
8,True,2015-08-09,2015/2016,Stoke,Liverpool,0,1,0.381274,0.329873,2015-08-09-Stoke-Liverpool,-0.425800,2.495550
9,True,2015-08-10,2015/2016,West Brom,Man City,0,3,0.435238,1.9242,2015-08-10-West Brom-Man City,-0.949027,2.824994


The dataset now contains a unique match identifier and goals and supremacy values based on asian handicap betting lines

### HOME ADVANTAGE FACTOR ###
In order to make predictions for future matches we need to identify the strength of each team, as well as introduce a metric that accounts for home advantage. As part of this process we also remove data that was played behind closed doors during the pandemic. Playing in stadiums without fans had an effect on home advantage, and altered the relationship between home team strength, away team strength and home advantage on the outcome of matches. Most leagues did not return to full stadiums until at least some way through the 2021/2022, so I have excluded all data from 1st March 2020 until the end of that season.

In [8]:
# Filters for past matches
fixtures_data_past = fixtures_data[fixtures_data['isResult'] == True].copy()

# Converts datatypes
to_float = ['xG_h', 'xG_a']
to_int = ['goals_h', 'goals_a']

fixtures_data_past[to_float] = fixtures_data_past[to_float].astype(float)
fixtures_data_past[to_int] = fixtures_data_past[to_int].astype(int)

# Removes matches played behind closed doors
covid_period_start = '2020-03-01'
covid_period_end = '2022-07-01'
                                                                                                         
fixtures_data_past = fixtures_data_past[(fixtures_data_past['Date'] < covid_period_start) | (fixtures_data_past['Date'] > covid_period_end)]                                                                                                                

# Finds average home and away goals over this period and defines home advantage factor 
fixtures_data_past_homeadv = pd.DataFrame({
    'homegoals': [fixtures_data_past['goals_h'].mean()],
    'awaygoals': [fixtures_data_past['goals_a'].mean()]
})

fixtures_data_past_homeadv['total_goals'] = fixtures_data_past_homeadv['homegoals'] + fixtures_data_past_homeadv['awaygoals']
fixtures_data_past_homeadv['homeadv'] = fixtures_data_past_homeadv['homegoals'] - fixtures_data_past_homeadv['awaygoals']
fixtures_data_past_homeadv['homeadv_pc'] = fixtures_data_past_homeadv['homeadv'] / fixtures_data_past_homeadv['total_goals']

home_adv = float(fixtures_data_past_homeadv['homeadv_pc'].iloc[0])
fixtures_data_past['HA'] = home_adv

In [9]:
season_counts = fixtures_data_past['season'].value_counts().sort_index()
print(season_counts)
print(home_adv)

season
2015/2016    380
2016/2017    380
2017/2018    380
2018/2019    380
2019/2020    276
2022/2023    380
2023/2024    380
2024/2025    380
2025/2026    309
Name: count, dtype: int64
0.10749891634156915


We now have a dataset that excludes all matches played behind closed doors and a home advantage factor we can use to make predictions for future matches

### TEAM PERFORMANCE DATAFRAME ###
Here we create a team focused dataframe that identifes team performace in each match based on goals, xG and team goal expectancies derived from betting lines

In [10]:
# Adds extra columns for analysis
fixtures_data_past['AsianHomeGoals'] = (fixtures_data_past['asian_sup'] + fixtures_data_past['asian_total_goals']) / 2
fixtures_data_past['AsianAwayGoals'] = fixtures_data_past['asian_total_goals'] - fixtures_data_past['AsianHomeGoals']
fixtures_data_past['total_goals'] = fixtures_data_past['goals_h'] + fixtures_data_past['goals_a']
fixtures_data_past['total_xG'] = fixtures_data_past['xG_h'] + fixtures_data_past['xG_a']

# Drops rows with missing odds data
fixtures_data_past = fixtures_data_past.dropna(subset = 'AsianHomeGoals')
fixtures_data_past = fixtures_data_past.dropna(subset = 'AsianAwayGoals')

# Creates team focused dataframe for team strength
fixtures_data_past_home = fixtures_data_past[['matchid','Date', 'Home', 'goals_h', 'goals_a', 'xG_h', 'xG_a', 'AsianHomeGoals', 'AsianAwayGoals', 'total_goals', 'total_xG', 'asian_total_goals']].copy()
fixtures_data_past_home['home_or_away'] = 'Home'

fixtures_data_past_away = fixtures_data_past[['matchid','Date', 'Away', 'goals_a', 'goals_h', 'xG_a', 'xG_h', 'AsianAwayGoals', 'AsianHomeGoals', 'total_goals', 'total_xG', 'asian_total_goals']].copy()
fixtures_data_past_away['home_or_away'] = 'Away'

fixtures_data_past_home = fixtures_data_past_home.rename(columns={'Home': 'Team', 
                                                                          'goals_h': 'GoalsScored', 'goals_a': 'GoalsConceded',
                                                                          'xG_h': 'xG_for', 'xG_a': 'xG_conc',
                                                                          'AsianHomeGoals': 'Asian_for', 'AsianAwayGoals': 'Asian_conc',
                                                                          })

fixtures_data_past_away = fixtures_data_past_away.rename(columns={'Away': 'Team', 
                                                                          'goals_a': 'GoalsScored', 'goals_h': 'GoalsConceded', 
                                                                          'xG_a': 'xG_for', 'xG_h': 'xG_conc',
                                                                          'AsianAwayGoals': 'Asian_for', 'AsianHomeGoals': 'Asian_conc',})


fixtures_data_team_strength = pd.concat([fixtures_data_past_home, fixtures_data_past_away], ignore_index=True)
fixtures_data_team_strength = fixtures_data_team_strength.sort_values(['Date', 'matchid'], ascending=False).reset_index()

In [11]:
fixtures_data_team_strength.head(10)

,index,matchid,Date,Team,GoalsScored,GoalsConceded,xG_for,xG_conc,Asian_for,Asian_conc,total_goals,total_xG,asian_total_goals,home_or_away
0,3214,2026-03-22-Tottenham-Nottm Forest,2026-03-22,Tottenham,0,3,1.618900,1.544080,1.369143,1.226672,3,3.162980,2.595816,Home
1,6429,2026-03-22-Tottenham-Nottm Forest,2026-03-22,Nottm Forest,3,0,1.544080,1.618900,1.226672,1.369143,3,3.162980,2.595816,Away
2,3212,2026-03-22-Newcastle-Sunderland,2026-03-22,Newcastle,1,2,1.267050,2.666570,1.971416,0.936777,3,3.933620,2.908194,Home
3,6427,2026-03-22-Newcastle-Sunderland,2026-03-22,Sunderland,2,1,2.666570,1.267050,0.936777,1.971416,3,3.933620,2.908194,Away
4,3213,2026-03-22-Aston Villa-West Ham,2026-03-22,Aston Villa,2,0,2.485280,0.760054,1.851150,1.057044,2,3.245334,2.908194,Home
5,6428,2026-03-22-Aston Villa-West Ham,2026-03-22,West Ham,0,2,0.760054,2.485280,1.057044,1.851150,2,3.245334,2.908194,Away
6,3211,2026-03-21-Leeds-Brentford,2026-03-21,Leeds,0,0,0.417296,0.302221,1.473370,1.280509,0,0.719517,2.753879,Home
7,6426,2026-03-21-Leeds-Brentford,2026-03-21,Brentford,0,0,0.302221,0.417296,1.280509,1.473370,0,0.719517,2.753879,Away
8,3209,2026-03-21-Fulham-Burnley,2026-03-21,Fulham,3,1,3.484380,1.925560,2.116349,0.936078,4,5.409940,3.052427,Home
9,6424,2026-03-21-Fulham-Burnley,2026-03-21,Burnley,1,3,1.925560,3.484380,0.936078,2.116349,4,5.409940,3.052427,Away


We now have a team focused dataframe with one row for each match per team, this can be used to create rolling averages of teams performance that we can use to create ratings to predict future performances.

### FINDING OPTIMAL PARAMETERS FOR ASSESSING TEAMS PERFORMANCE ###
To create predictions for future matches I want to derive expected goals and supremacy values for each match for the rest of the season. To derive these I will use team attack strength and defence strength ratings to predict the number of goals scored by each team in each match. To get these predictions I will use the formulae:

#### Home_Pred_Goals = ((Home_AS * Away_DS) / DS_avg) + (HA_factor / 2) ####
#### Away_Pred_Goals = ((Away_AS* Home_DS ) / DS_avg) - (HA_factor / 2) ####

Home_AS, Away_AS, Home_DS and Away_DS are measures of the past attack and defense performance for each team, based on historic goals, xG performaces and betting lines.

DS_avg is the average amount of goals conceded per match by a single team over the same time period, HA_factor is the home advantage factor we identified earlier.

In order to optimise the attack and defence strength values for each team I want to find the best combination of n_matches, goals, xG and betting line ratings. This function tests a range of n_matches and weighted combinations of each strength measure to find the optimal combination of these parameters.

I am assessing each combination by measuring the mean absolute error of the predicted goals and supremacy for each match compared to asian_total_goals and asian_sup. These values are derived from historic betting lines and are assumed to be efficient at KO. 

In [12]:
# Splits data into train and test sets
fixtures_data_past_train = fixtures_data_past[fixtures_data_past['Date'] < '2023-08-01'].copy()
fixtures_data_past_test = fixtures_data_past[fixtures_data_past['Date'] > '2023-08-01'].copy()

# Defines ranges for parameters
n_matches_range = range(5, 30)
goals_wgt_range = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]
xG_wgt_range = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]
asian_wgt_range = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]

cols_to_avg_team = ['GoalsScored', 'GoalsConceded', 'xG_for', 'xG_conc', 'Asian_for', 'Asian_conc']
cols_to_avg_league = ['total_goals', 'total_xG', 'asian_total_goals']

# Function to find maes of different parameters
def calculate_predictions(n_matches, goals_wgt, xG_wgt, asian_wgt, fixtures, team_strength):

    if goals_wgt + xG_wgt + asian_wgt != 1:
        raise ValueError("Weights must sum to 1.")
    
    # Team averages for each n
    for col in cols_to_avg_team:
        team_strength[f'{col}_roll'] = (
            team_strength
            .groupby('Team', group_keys=False)[col]
            .transform(lambda x: x.shift(-1)[::-1].rolling(window=n_matches, min_periods=n_matches).mean()[::-1])
        )
    # League averages for each n * n_teams_div
    for col in cols_to_avg_league:
        team_strength[f'{col}_roll'] = (
            team_strength[col]
            .transform(lambda x: x.shift(-1)[::-1].rolling(window=(n_matches * n_teams_div), min_periods=(n_matches * n_teams_div)).mean()[::-1])
        )
    
    # Merges for team strength with past fixtures for home team
    fixtures = fixtures.merge(
        team_strength[team_strength['home_or_away'] == 'Home'][['matchid', 'Team', 'GoalsScored_roll', 'GoalsConceded_roll', 'xG_for_roll', 'xG_conc_roll', 'Asian_for_roll', 'Asian_conc_roll', 'total_goals_roll', 'total_xG_roll', 'asian_total_goals_roll']],
        left_on=['Home', 'matchid'],
        right_on=['Team', 'matchid'],
        how='left'
    ).rename(
        columns={'GoalsScored_roll': 'Home_GoalsScored_roll', 'GoalsConceded_roll': 'Home_GoalsConceded_roll', 
                 'xG_for_roll': 'Home_xG_for_roll', 'xG_conc_roll': 'Home_xG_conc_roll', 
                 'Asian_for_roll': 'Home_Asian_for_roll', 'Asian_conc_roll': 'Home_Asian_conc_roll'}
    ).drop(columns=['Team'])
    
    # Merges for team strength with past fixtures for away team
    fixtures = fixtures.merge(
        team_strength[team_strength['home_or_away'] == 'Away'][['matchid', 'Team', 'GoalsScored_roll', 'GoalsConceded_roll', 'xG_for_roll', 'xG_conc_roll', 'Asian_for_roll', 'Asian_conc_roll']],
        left_on=['Away', 'matchid'],
        right_on=['Team', 'matchid'],
        how='left'
    ).rename(
        columns={'GoalsScored_roll': 'Away_GoalsScored_roll', 'GoalsConceded_roll': 'Away_GoalsConceded_roll', 
                 'xG_for_roll': 'Away_xG_for_roll', 'xG_conc_roll': 'Away_xG_conc_roll', 
                 'Asian_for_roll': 'Away_Asian_for_roll', 'Asian_conc_roll': 'Away_Asian_conc_roll'}
    ).drop(columns=['Team'])
    
    # Calculates av goals per team
    fixtures['avg_ds_goals'] = fixtures['total_goals_roll'] / 2

    # Drops any rows with missing rolling avs due to small sample size
    fixtures = fixtures.dropna()
    
    # Calculates weighted home, away team AS and DS based on weights
    fixtures['weight_AS_home'] = (
        fixtures['Home_GoalsScored_roll'] * goals_wgt +
        fixtures['Home_xG_for_roll'] * xG_wgt +
        fixtures['Home_Asian_for_roll'] * asian_wgt
    )
    
    fixtures['weight_DS_home'] = (
        fixtures['Home_GoalsConceded_roll'] * goals_wgt +
        fixtures['Home_xG_conc_roll'] * xG_wgt +
        fixtures['Home_Asian_conc_roll'] * asian_wgt
    )
    
    fixtures['weight_AS_away'] = (
        fixtures['Away_GoalsScored_roll'] * goals_wgt +
        fixtures['Away_xG_for_roll'] * xG_wgt +
        fixtures['Away_Asian_for_roll'] * asian_wgt
    )
    
    fixtures['weight_DS_away'] = (
        fixtures['Away_GoalsConceded_roll'] * goals_wgt +
        fixtures['Away_xG_conc_roll'] * xG_wgt +
        fixtures['Away_Asian_conc_roll'] * asian_wgt
    )
    
    # Calculates predictions based on this combination of weights
    fixtures['home_pred'] = (fixtures['weight_AS_home'] * fixtures['weight_DS_away']) / fixtures['avg_ds_goals']
    fixtures['away_pred'] = (fixtures['weight_AS_away'] * fixtures['weight_DS_home']) / fixtures['avg_ds_goals']
    fixtures['total_pred'] = fixtures['home_pred'] + fixtures['away_pred']
    fixtures['sup_pred'] = fixtures['home_pred'] - fixtures['away_pred']
    fixtures['Match_HA'] = fixtures['total_pred'] * fixtures['HA']
    fixtures['home_pred'] = fixtures['home_pred'] + (fixtures['Match_HA'] / 2)
    fixtures['away_pred'] = fixtures['away_pred'] - (fixtures['Match_HA'] / 2)
    fixtures['total_pred'] = fixtures['home_pred'] + fixtures['away_pred']
    fixtures['sup_pred'] = fixtures['home_pred'] - fixtures['away_pred']
    
    # Calculates MAE against asian lines for each match
    mae_total = mean_absolute_error(fixtures['total_pred'], fixtures['asian_total_goals'])
    mae_sup = mean_absolute_error(fixtures['sup_pred'], fixtures['asian_sup'])
    
    return mae_total, mae_sup



### DEFINING BEST PARAMETERS AND TESTING ###
In this section we run the function on a range of combination of weights and n matches and record the results of the mae for goals and supremacy in a dataframe. The parameters with the lowest weighted mae are identified and tested against the test dataset.

In [13]:
# Creates initial list
results = []

# Iterates through all combos and creates dataframe to store results
for n_matches, goals_wgt, xG_wgt, asian_wgt in product(n_matches_range, goals_wgt_range, xG_wgt_range, asian_wgt_range):
    try:
        mae_total, mae_sup = calculate_predictions(n_matches, goals_wgt, xG_wgt, asian_wgt, fixtures_data_past_train.copy(), fixtures_data_team_strength.copy())
        
        results.append({
            'n_matches': n_matches,
            'goals_wgt': goals_wgt,
            'xG_wgt': xG_wgt,
            'asian_wgt': asian_wgt,
            'mae_total': mae_total,
            'mae_sup': mae_sup
        })
    except ValueError:
        continue  
    
results_df = pd.DataFrame(results)

# Preference for finding sup diffs
results_df['comb_mae'] = (results_df['mae_total']*0.25) + (results_df['mae_sup']*0.75)
results_df = results_df.sort_values('comb_mae').reset_index()

best_params = [results_df['n_matches'].iloc[0], results_df['goals_wgt'].iloc[0], results_df['xG_wgt'].iloc[0], results_df['asian_wgt'].iloc[0]]

# Test params against test data
mae_total_test, mae_sup_test = calculate_predictions(
    n_matches=best_params[0],
    goals_wgt=best_params[1],
    xG_wgt=best_params[2],
    asian_wgt=best_params[3],
    fixtures=fixtures_data_past_test.copy(),
    team_strength=fixtures_data_team_strength.copy()
)


In [14]:
results_df.head(10)

,index,n_matches,goals_wgt,xG_wgt,asian_wgt,mae_total,mae_sup,comb_mae
0,352,16,0.1,0.1,0.8,0.219370,0.233962,0.230314
1,449,19,0.1,0.2,0.7,0.220644,0.233775,0.230493
2,384,17,0.1,0.1,0.8,0.219450,0.234542,0.230769
3,424,18,0.2,0.1,0.7,0.215576,0.235914,0.230829
4,456,19,0.2,0.1,0.7,0.214013,0.236480,0.230863
5,417,18,0.1,0.2,0.7,0.221612,0.234253,0.231093
6,385,17,0.1,0.2,0.7,0.221875,0.234393,0.231263
7,416,18,0.1,0.1,0.8,0.219532,0.235296,0.231355
8,353,16,0.1,0.2,0.7,0.221669,0.234588,0.231359
9,448,19,0.1,0.1,0.8,0.218608,0.235806,0.231506


In [15]:
print(best_params)

print(mae_total_test, mae_sup_test)

[16, 0.1, 0.1, 0.8]
0.2533723134126499 0.2493446510286991


We have now identified the combination of n_matches goals_wgt, xG_wgt and asian_wgt that have historically minimised the error between our predicted goals and supremacy and the goals and supremacy implied by the betting lines for each match. We will use the parameters to make future predictions.

### APPLYING OPTIMAL PARAMETERS TO FUTURE FIXTURES ###
In this section we use the optimal params identified above to get the ratings to predict future matches. We then use these ratings combined with our home advantage factor to make predictions for every match for the rest of the season.

In [16]:
### Current team averages
current_team_strength = fixtures_data_team_strength.copy()
best_n_matches = best_params[0]

# Team averages for best_n_matches
for col in cols_to_avg_team:
    current_team_strength[f'{col}_roll'] = (
    current_team_strength
   .groupby('Team', group_keys=False)[col]
   .transform(lambda x: x[::-1].rolling(window=best_n_matches, min_periods=best_n_matches).mean()[::-1])
)

# League averages for best_n_matches * n_teams_div
for col in cols_to_avg_league:
    current_team_strength[f'{col}_roll'] = (
    current_team_strength[col]
    .transform(lambda x: x[::-1].rolling(window=(best_n_matches * n_teams_div), min_periods=(best_n_matches * n_teams_div)).mean()[::-1])
)

# Filters for most recent ratings
current_team_strength = current_team_strength.drop_duplicates(subset='Team', keep='first')

# Filters for this seasons teams
current_season_teams = current_team_strength[(current_team_strength['Date'] > '2025-07-01')]
current_season_teams = current_season_teams['Team'].unique()
current_team_strength = current_team_strength[current_team_strength['Team'].isin(current_season_teams)]

# Filters relevant columns
current_team_AS_DS = current_team_strength[['Team', 'GoalsScored_roll', 'xG_for_roll', 'Asian_for_roll', 'GoalsConceded_roll', 'xG_conc_roll', 'Asian_conc_roll', 'total_goals_roll']].copy()

# Assigns weights based on optimal params
goals_wgt = best_params[1]
xG_wgt = best_params[2]
asian_wgt = best_params[3]

# Applies weights to get optimised team AS and DS ratings
current_team_AS_DS['weight_AS'] = (current_team_AS_DS['GoalsScored_roll'] * goals_wgt) + (current_team_AS_DS['xG_for_roll'] * xG_wgt) + (current_team_AS_DS['Asian_for_roll'] * asian_wgt)
current_team_AS_DS['weight_DS'] = (current_team_AS_DS['GoalsConceded_roll'] * goals_wgt) + (current_team_AS_DS['xG_conc_roll'] * xG_wgt) + (current_team_AS_DS['Asian_conc_roll'] * asian_wgt)
current_team_AS_DS['DS_avg'] = current_team_AS_DS['weight_DS'].mean()

# Upcoming fixtures
fixtures_data_future = fixtures_data[fixtures_data['isResult'] == False].copy()
fixtures_data_future = fixtures_data_future[fixtures_data_future['Date'] > '2025-07-01']
fixtures_data_future = fixtures_data_future[['Date', 'Home', 'Away', ]]

# Adds AS and DS to upcoming fixtures
fixtures_data_future = fixtures_data_future.merge(
    current_team_AS_DS[['Team', 'weight_AS', 'weight_DS', 'DS_avg']],
    left_on='Home',
    right_on='Team',
    how='left'
).rename(columns={'weight_AS': 'Home_AS', 'weight_DS': 'Home_DS'}).drop(columns=['Team'])

fixtures_data_future = fixtures_data_future.merge(
    current_team_AS_DS[['Team', 'weight_AS', 'weight_DS']],
    left_on='Away',
    right_on='Team',
    how='left'
).rename(columns={'weight_AS': 'Away_AS', 'weight_DS': 'Away_DS'}).drop(columns=['Team'])

# Predicted match goals and supremacy for rest of the season
fixtures_data_future['HA'] = home_adv
fixtures_data_future['Home_Pred_Goals'] = (fixtures_data_future['Home_AS'] * fixtures_data_future['Away_DS']) / fixtures_data_future['DS_avg']
fixtures_data_future['Away_Pred_Goals'] = (fixtures_data_future['Away_AS'] * fixtures_data_future['Home_DS']) / fixtures_data_future['DS_avg']
fixtures_data_future['Total_Pred_Goals'] = fixtures_data_future['Home_Pred_Goals'] + fixtures_data_future['Away_Pred_Goals']
fixtures_data_future['Pred_Sup'] = fixtures_data_future['Home_Pred_Goals'] - fixtures_data_future['Away_Pred_Goals']

# Adds home advantage factor
fixtures_data_future['Total_HA'] = fixtures_data_future['Total_Pred_Goals'] * fixtures_data_future['HA']
fixtures_data_future['Home_Pred_Goals'] = fixtures_data_future['Home_Pred_Goals'] + (fixtures_data_future['Total_HA'] / 2)
fixtures_data_future['Away_Pred_Goals'] = fixtures_data_future['Away_Pred_Goals'] - (fixtures_data_future['Total_HA'] / 2)
fixtures_data_future['Total_Pred_Goals'] = fixtures_data_future['Home_Pred_Goals'] + fixtures_data_future['Away_Pred_Goals']
fixtures_data_future['Pred_Sup'] = fixtures_data_future['Home_Pred_Goals'] - fixtures_data_future['Away_Pred_Goals']
fixtures_data_future.drop(['Home_AS', 'Home_DS', 'Home_DS', 'DS_avg', 'Away_AS', 'Away_DS', 'HA', 
                    'Total_HA'], axis = 1, inplace=True)

current_team_AS_DS.drop(['GoalsScored_roll', 'xG_for_roll', 'Asian_for_roll', 'GoalsConceded_roll', 'xG_conc_roll', 'Asian_conc_roll', 'total_goals_roll', 
                    'DS_avg'], axis = 1, inplace=True)

In [17]:
current_team_AS_DS.head(20).sort_values('weight_AS', ascending=[False])

,Team,weight_AS,weight_DS
27,Man City,2.090822,1.053080
34,Arsenal,2.053550,0.789641
13,Liverpool,1.928347,1.148104
11,Chelsea,1.821156,1.335436
15,Man Utd,1.766169,1.308884
2,Newcastle,1.618525,1.438556
14,Bournemouth,1.537748,1.581442
4,Aston Villa,1.477623,1.404442
7,Brentford,1.466130,1.322599
12,Brighton,1.389926,1.404934


In [18]:
fixtures_data_future.head(10)

,Date,Home,Away,Home_Pred_Goals,Away_Pred_Goals,Total_Pred_Goals,Pred_Sup
0,2026-04-10,West Ham,Wolverhampton,1.814407,1.025459,2.839866,0.788948
1,2026-04-11,Arsenal,Bournemouth,2.429802,0.678936,3.108737,1.750866
2,2026-04-11,Brentford,Everton,1.597470,0.941019,2.538489,0.656451
3,2026-04-11,Burnley,Brighton,1.096770,1.700439,2.797209,-0.603670
4,2026-04-11,Liverpool,Fulham,2.087664,0.941971,3.029635,1.145692
5,2026-04-12,Nottm Forest,Aston Villa,1.343210,1.349675,2.692885,-0.006464
6,2026-04-12,Sunderland,Tottenham,1.310532,1.186224,2.496756,0.124309
7,2026-04-12,Crystal Palace,Newcastle,1.413232,1.458371,2.871603,-0.045138
8,2026-04-12,Chelsea,Man City,1.512607,1.769021,3.281628,-0.256413
9,2026-04-13,Man Utd,Leeds,1.924674,0.935393,2.860067,0.989281


The first table show the curent weighted attack and defence ratings for each team in the league. These are used to make the predictions for upcoming games in the second table.

### ASSESSING DRAW FACTOR IN HISTORICAL DATA ###

To make predictions for future matches we will use a poisson distribution to predict the outcome of each match based on their predicted goals identified above. One weakness of using a poisson distribution is that it assumes goals are statistically independent of each other. In reality goals scored by teams in any match will be influenced by the current score and overall gamestate. Historically an unrefined poisson model has underestimated draws and low scoring scorelines in general.

This section compares the proportion of draws observed in  historical data with how many draws would be predicted by a standard poisson model. It also shows the difference between poisson predicted scorelines and observed scorelines in low scoring matches.

In [19]:
# Observed scorelines for all past matches
scorelines = [(home_goals, away_goals) for home_goals in range(11) for away_goals in range(11)]

total_matches = len(fixtures_data_past_train)

observed_probs = {
    scoreline: len(fixtures_data_past_train[(fixtures_data_past_train['goals_h'] == scoreline[0]) & (fixtures_data_past_train['goals_a'] == scoreline[1])]) / total_matches
    for scoreline in scorelines
}

# Observed scoreline probs matrix
max_goals = 11

observed_probs_matrix = np.zeros((max_goals, max_goals))

for (home_goals, away_goals), prob in observed_probs.items():
    observed_probs_matrix[home_goals, away_goals] = prob

observed_probs_matrix = pd.DataFrame(observed_probs_matrix)

# Calculate predicted scorelines for all past matches
predicted_probs = {scoreline: 0 for scoreline in scorelines}

for _, row in fixtures_data_past_train.iterrows():
    home_exp = row['AsianHomeGoals']
    away_exp = row['AsianAwayGoals']

    for scoreline in scorelines:
        home_goals, away_goals = scoreline
        prob = poisson.pmf(home_goals, home_exp) * poisson.pmf(away_goals, away_exp)
        predicted_probs[scoreline] += prob

# Average scoreline probabilities
predicted_probs = {scoreline: prob / len(fixtures_data_past_train) for scoreline, prob in predicted_probs.items()}

# Predicted scoreline probs matrix
max_goals = 11

predicted_probs_matrix = np.zeros((max_goals, max_goals))

for (home_goals, away_goals), prob in predicted_probs.items():
    predicted_probs_matrix[home_goals, away_goals] = prob

predicted_probs_matrix = pd.DataFrame(predicted_probs_matrix)

scorelinediffs = predicted_probs_matrix - observed_probs_matrix

# Predicted outcome probabilities
predicted_outcomes = {"home_win": 0, "draw": 0, "away_win": 0}

for scoreline, prob in predicted_probs.items():
    home_goals, away_goals = scoreline
    
    if home_goals > away_goals:  
        predicted_outcomes["home_win"] += prob
    elif home_goals == away_goals:  
        predicted_outcomes["draw"] += prob
    else:
        predicted_outcomes["away_win"] += prob

# Observed outcomes probabilities
observed_outcomes = {"home_win": 0, "draw": 0, "away_win": 0}

for _, row in fixtures_data_past_train.iterrows():
    if row['goals_h'] > row['goals_a']: 
        observed_outcomes["home_win"] += 1
    elif row['goals_h'] == row['goals_a']:
        observed_outcomes["draw"] += 1
    else:
        observed_outcomes["away_win"] += 1

observed_outcomes = {key: value / total_matches for key, value in observed_outcomes.items()}


In [20]:
print(predicted_outcomes)

print(observed_outcomes)

print(scorelinediffs.iloc[:3, :3])


{'home_win': 0.4512711243586006, 'draw': 0.2268053421721456, 'away_win': 0.32188761225153795}
{'home_win': 0.46042830540037244, 'draw': 0.2378957169459963, 'away_win': 0.3016759776536313}
          0         1         2
0 -0.004242  0.014468  0.005112
1 -0.004384  0.001584 -0.004061
2 -0.006960 -0.003153 -0.009539


Here we can see how 1X2 outcomes predicted by a poisson model compare to our observed outcomes. It also shows differences in observed scorelines and poisson predicted scorelines in low scoring matches.

### CALCULATING A DRAW ADJUSTMENT FACTOR ###
This section applies the Dixon-Coles low-score correction. Rho adjusts the four scorelines where goals are most correlated (0-0, 1-0, 0-1, 1-1), leaving every other scoreline at its raw poisson probability. Rho is fitted by maximum likelihood on the training set's actual scorelines, so the adjustment reflects the historical relationship between low scoring games and draws directly.

In [29]:
# Dixon-Coles low-score correction factor
def dixon_coles_tau(home_goals, away_goals, home_exp, away_exp, rho):
    tau = np.ones_like(np.asarray(home_exp, dtype=float))
    tau = np.where((home_goals == 0) & (away_goals == 0), 1 - (home_exp * away_exp * rho), tau)
    tau = np.where((home_goals == 0) & (away_goals == 1), 1 + (home_exp * rho), tau)
    tau = np.where((home_goals == 1) & (away_goals == 0), 1 + (away_exp * rho), tau)
    tau = np.where((home_goals == 1) & (away_goals == 1), 1 - rho, tau)
    return tau

# The four scorelines tau adjusts
low_scorelines = [(0, 0), (0, 1), (1, 0), (1, 1)]

# Negative log-likelihood of the training set's actual scorelines
def dixon_coles_neg_log_likelihood(rho):
    home_goals = fixtures_data_past_train['goals_h']
    away_goals = fixtures_data_past_train['goals_a']
    home_exp = fixtures_data_past_train['AsianHomeGoals']
    away_exp = fixtures_data_past_train['AsianAwayGoals']

    tau = dixon_coles_tau(home_goals, away_goals, home_exp, away_exp, rho)
    probs = tau * poisson.pmf(home_goals, home_exp) * poisson.pmf(away_goals, away_exp)

    return -np.log(np.clip(probs, 1e-10, None)).sum()

# Fits rho by maximising the likelihood of the training set's actual scorelines
rho_fit = minimize_scalar(dixon_coles_neg_log_likelihood, bounds=(-1, 1), method='bounded')
rho = rho_fit.x

# Function to calculate probabilities with Dixon-Coles adjusted scorelines for matches in training set
def calculate_adjusted_probabilities(row):
    home_goal_expectation = row['AsianHomeGoals']
    away_goal_expectation = row['AsianAwayGoals']

    home_probs = poisson.pmf(range(11), home_goal_expectation)
    away_probs = poisson.pmf(range(11), away_goal_expectation)

    probability_matrix = np.outer(home_probs, away_probs)

    adjusted_probability_matrix = probability_matrix.copy()
    for home_goals, away_goals in low_scorelines:
        tau = dixon_coles_tau(home_goals, away_goals, home_goal_expectation, away_goal_expectation, rho)
        adjusted_probability_matrix[home_goals, away_goals] *= tau

    adjusted_probability_matrix /= adjusted_probability_matrix.sum()

    home = np.sum(np.tril(adjusted_probability_matrix, -1))
    draw = np.sum(np.diag(adjusted_probability_matrix))
    away = np.sum(np.triu(adjusted_probability_matrix, 1))

    return pd.Series([home, draw, away], index=['home_pc', 'draw_pc', 'away_pc'])

fixtures_data_past_train[['home_pc', 'draw_pc', 'away_pc']] = fixtures_data_past_train.apply(calculate_adjusted_probabilities, axis=1)
fixtures_data_past_train[['home_win_price', 'draw_price', 'away_win_price']] = 1 / fixtures_data_past_train[['home_pc', 'draw_pc', 'away_pc']]
fixtures_data_past_train = fixtures_data_past_train.round(2)

# Sense checking adjusted predicted outcomes
predicted_outcomes_adjusted = {"home_win": 0, "draw": 0, "away_win": 0}
predicted_outcomes_adjusted['home_win'] = fixtures_data_past_train['home_pc'].sum() / len(fixtures_data_past_train)
predicted_outcomes_adjusted['draw'] = fixtures_data_past_train['draw_pc'].sum() / len(fixtures_data_past_train)
predicted_outcomes_adjusted['away_win'] = fixtures_data_past_train['away_pc'].sum() / len(fixtures_data_past_train)

In [22]:
print(predicted_outcomes)

print(predicted_outcomes_adjusted)

print(observed_outcomes)

{'home_win': 0.4512711243586006, 'draw': 0.2268053421721456, 'away_win': 0.32188761225153795}
{'home_win': 0.44649906890130353, 'draw': 0.23633147113594047, 'away_win': 0.31711824953445067}
{'home_win': 0.46042830540037244, 'draw': 0.2378957169459963, 'away_win': 0.3016759776536313}


This shows the effect that the scoreline adjustments have made in getting our 1X2 predictions closer to observed historical results.

### CURRENT SEASON TABLE ###

This section creates a league table based on results so far this season

In [23]:
# Sets up current table 
current_season_table = pd.DataFrame(columns=['Team','MP', 'W','D','L','GF','GA','GD','Pts'])
current_season_table['Team'] = current_season_teams
current_season_table[['MP', 'W', 'D', 'L', 'GF', 'GA', 'GD', 'Pts']] = 0

currentseason_past = fixtures_data[fixtures_data['Date'] > '2025-07-01'].copy()
currentseason_past = currentseason_past[currentseason_past['isResult'] == True]
currentseason_past[to_float] = currentseason_past[to_float].astype(float)
currentseason_past[to_int] = currentseason_past[to_int].astype(int)

# Adds values to table based on results
for index, row in currentseason_past.iterrows():
    home, away = row['Home'], row['Away']
    goals_h, goals_a = row['goals_h'], row['goals_a']

    home_index = current_season_table[current_season_table['Team'] == home].index[0]
    away_index = current_season_table[current_season_table['Team'] == away].index[0]
    
    current_season_table.at[home_index, 'MP'] += 1
    current_season_table.at[away_index, 'MP'] += 1

    if goals_h > goals_a:
        current_season_table.at[home_index, 'Pts'] += 3
        current_season_table.at[home_index, 'W'] += 1
        current_season_table.at[away_index, 'L'] += 1
    elif goals_h < goals_a: 
        current_season_table.at[away_index, 'Pts'] += 3
        current_season_table.at[away_index, 'W'] += 1
        current_season_table.at[home_index, 'L'] += 1
    else: 
        current_season_table.at[home_index, 'Pts'] += 1
        current_season_table.at[away_index, 'Pts'] += 1
        current_season_table.at[home_index, 'D'] += 1
        current_season_table.at[away_index, 'D'] += 1

    current_season_table.at[home_index, 'GF'] += goals_h
    current_season_table.at[home_index, 'GA'] += goals_a
    current_season_table.at[home_index, 'GD'] += (goals_h - goals_a)
    
    current_season_table.at[away_index, 'GF'] += goals_a
    current_season_table.at[away_index, 'GA'] += goals_h
    current_season_table.at[away_index, 'GD'] += (goals_a - goals_h)


current_season_table = current_season_table.sort_values(['Pts', 'GD', 'GF'], ascending=False)

In [24]:
current_season_table

,Team,MP,W,D,L,GF,GA,GD,Pts
19,Arsenal,31,21,7,3,61,22,39,70
18,Man City,30,18,7,5,60,28,32,61
15,Man Utd,31,15,10,6,56,43,13,55
4,Aston Villa,31,16,6,9,42,37,5,54
13,Liverpool,31,14,7,10,50,42,8,49
11,Chelsea,31,13,9,9,53,38,15,48
7,Brentford,31,13,7,11,46,42,4,46
10,Everton,31,13,7,11,37,35,2,46
8,Fulham,31,13,5,13,43,44,-1,44
12,Brighton,31,11,10,10,41,37,4,43


### END OF SEASON TABLE PREDICTION ###

This section makes predictions for all future matches based on the predicted goals for each match. It adjusts the probability of each outcome to account for the increased prevalence of draws and low scoring matches. It then sums the predicted number of wins, draw, losses and goals for each team, adds these values to the current table and sorts to give a prediction of what the league table will look like at the end of the season.

In [25]:
# Function to get 1X2 predictions for future matches, with Dixon-Coles adjustments for low scoring draws
def calculate_adjusted_probabilities_future(row):
    home_goal_expectation = row['Home_Pred_Goals']
    away_goal_expectation = row['Away_Pred_Goals']
    
    home_probs = poisson.pmf(range(11), home_goal_expectation)
    away_probs = poisson.pmf(range(11), away_goal_expectation)
    
    probability_matrix = np.outer(home_probs, away_probs)
    
    adjusted_probability_matrix = probability_matrix.copy()
    for home_goals, away_goals in low_scorelines:
        tau = dixon_coles_tau(home_goals, away_goals, home_goal_expectation, away_goal_expectation, rho)
        adjusted_probability_matrix[home_goals, away_goals] *= tau
    
    adjusted_probability_matrix /= adjusted_probability_matrix.sum()
    
    home = np.sum(np.tril(adjusted_probability_matrix, -1))  
    draw = np.sum(np.diag(adjusted_probability_matrix))     
    away = np.sum(np.triu(adjusted_probability_matrix, 1))
    
    return pd.Series([home, draw, away], index=['home_pc', 'draw_pc', 'away_pc'])

fixtures_data_future[['home_pc', 'draw_pc', 'away_pc']] = fixtures_data_future.apply(calculate_adjusted_probabilities_future, axis=1)
fixtures_data_future[['home_win_price', 'draw_price', 'away_win_price']] = 1 / fixtures_data_future[['home_pc', 'draw_pc', 'away_pc']]

# Sets up future table prediction
current_season_table_predictions = current_season_table.copy()
columns_to_float = ['Pts', 'W', 'D', 'L', 'GF', 'GA', 'GD']
current_season_table_predictions[columns_to_float] = current_season_table_predictions[columns_to_float].astype(float)

# Adds points, goals and matches played to current table
for index, row in fixtures_data_future.iterrows():
    home, away = row['Home'], row['Away']
    home_pc, draw_pc, away_pc, Exp_HG, Exp_AG = row['home_pc'], row['draw_pc'], row['away_pc'], row['Home_Pred_Goals'], row['Away_Pred_Goals']

    home_index = current_season_table_predictions[current_season_table_predictions['Team'] == home].index[0]
    away_index = current_season_table_predictions[current_season_table_predictions['Team'] == away].index[0]
    
    current_season_table_predictions.at[home_index, 'MP'] += 1
    
    current_season_table_predictions.at[home_index, 'Pts'] += 3 * home_pc
    current_season_table_predictions.at[home_index, 'Pts'] += 1 * draw_pc
    
    current_season_table_predictions.at[home_index, 'W'] += 1 * home_pc
    current_season_table_predictions.at[home_index, 'D'] += 1 * draw_pc
    current_season_table_predictions.at[home_index, 'L'] += 1 * away_pc
    
    current_season_table_predictions.at[away_index, 'MP'] += 1
    
    current_season_table_predictions.at[away_index, 'Pts'] += 3 * away_pc
    current_season_table_predictions.at[away_index, 'Pts'] += 1 * draw_pc
    
    current_season_table_predictions.at[away_index, 'W'] += 1 * away_pc
    current_season_table_predictions.at[away_index, 'D'] += 1 * draw_pc
    current_season_table_predictions.at[away_index, 'L'] += 1 * home_pc
    
    current_season_table_predictions.at[home_index, 'GF'] += Exp_HG
    current_season_table_predictions.at[home_index, 'GA'] += Exp_AG
    current_season_table_predictions.at[home_index, 'GD'] += (Exp_HG - Exp_AG)
    
    current_season_table_predictions.at[away_index, 'GF'] += Exp_AG
    current_season_table_predictions.at[away_index, 'GA'] += Exp_HG
    current_season_table_predictions.at[away_index, 'GD'] += (Exp_AG - Exp_HG)

current_season_table_predictions = current_season_table_predictions.sort_values(['Pts', 'GD', 'GF'], ascending=False)

current_season_table_predictions = current_season_table_predictions.round(2)

In [26]:
current_season_table_predictions

,Team,MP,W,D,L,GF,GA,GD,Pts
19,Arsenal,38,25.78,8.30,3.91,76.36,27.37,48.99,85.65
18,Man City,38,22.67,8.71,6.63,76.25,36.69,39.56,76.70
15,Man Utd,38,18.31,11.67,8.01,68.01,52.03,15.98,66.62
4,Aston Villa,38,18.79,7.70,11.52,52.30,46.81,5.49,64.06
13,Liverpool,38,17.77,8.61,11.63,63.15,50.10,13.04,61.90
11,Chelsea,38,16.23,10.61,11.16,65.29,47.74,17.55,59.30
7,Brentford,38,15.68,8.68,13.63,55.87,51.89,3.98,55.74
10,Everton,38,15.04,8.85,14.11,44.93,45.34,-0.41,53.97
12,Brighton,38,13.84,11.75,12.40,51.50,46.48,5.01,53.28
8,Fulham,38,15.25,6.66,16.09,52.09,55.16,-3.07,52.42


This is a prediction of how the league table will look at the end of the season.

### MONTE CARLO SIMULATION OF REST OF SEASON ###
This section runs a Monte Carlo simulation to calculate the likelihood of each team finishing 1st, top 4, top 6, Top 8 and bottom 3 in the league at the end of the season.

In [27]:
# Defines simulation variables
N = 100
teams = current_season_teams
num_teams = len(teams)

results = pd.DataFrame(0, index=teams, columns=['1st', 'Top 4', 'Top 6', 'Top 8', 'Bottom 3'])
results.index.name = "Team"

# Function to simulate a single match
def simulate_match(home_pc, draw_pc, away_pc):
    outcome = np.random.choice(['home', 'draw', 'away'], p=[home_pc, draw_pc, away_pc])
    if outcome == 'home':
        return 3, 0  
    elif outcome == 'draw':
        return 1, 1  
    else:
        return 0, 3  

pts_table = current_season_table[['Team', 'Pts']].copy()
pts_table['Pts']  = pts_table['Pts'].astype(float)

# Runs the simulations
for sim in range(N):

    sim_table = pts_table.copy()
    
    # Simulates remaining matches and add predicted points to league table
    for _, row in fixtures_data_future.iterrows():
        home, away = row['Home'], row['Away']
        home_pc, draw_pc, away_pc = row['home_pc'], row['draw_pc'], row['away_pc']
        
        home_pts, away_pts = simulate_match(home_pc, draw_pc, away_pc)
        
        home_index = sim_table[sim_table['Team'] == home].index[0]
        sim_table.at[home_index, 'Pts'] += home_pts

        away_index = sim_table[sim_table['Team'] == away].index[0]
        sim_table.at[away_index, 'Pts'] += away_pts

    # Sorts table
    sim_table = sim_table.sort_values('Pts', ascending=False).reset_index(drop=True)
    
    # Adds 1 to results for each sim in each bucket for each team
    results.loc[sim_table.iloc[0]['Team'], '1st'] += 1
    for i, team in enumerate(sim_table['Team']):
        if i < 4:
            results.loc[team, 'Top 4'] += 1
        if i < 6:
            results.loc[team, 'Top 6'] += 1
        if i < 8:
            results.loc[team, 'Top 8'] += 1
        if i >= num_teams - 3:
            results.loc[team, 'Bottom 3'] += 1

league_predictions = results / N
table_predictions_probs = pd.merge(current_season_table_predictions, league_predictions, on=["Team"], how="left")

In [28]:
table_predictions_probs

,Team,MP,W,D,L,GF,GA,GD,Pts,1st,Top 4,Top 6,Top 8,Bottom 3
0,Arsenal,38,25.78,8.30,3.91,76.36,27.37,48.99,85.65,0.94,1.00,1.00,1.00,0.00
1,Man City,38,22.67,8.71,6.63,76.25,36.69,39.56,76.70,0.06,1.00,1.00,1.00,0.00
2,Man Utd,38,18.31,11.67,8.01,68.01,52.03,15.98,66.62,0.00,0.90,1.00,1.00,0.00
3,Aston Villa,38,18.79,7.70,11.52,52.30,46.81,5.49,64.06,0.00,0.66,0.98,1.00,0.00
4,Liverpool,38,17.77,8.61,11.63,63.15,50.10,13.04,61.90,0.00,0.31,0.89,0.97,0.00
5,Chelsea,38,16.23,10.61,11.16,65.29,47.74,17.55,59.30,0.00,0.11,0.74,0.93,0.00
6,Brentford,38,15.68,8.68,13.63,55.87,51.89,3.98,55.74,0.00,0.00,0.09,0.57,0.00
7,Everton,38,15.04,8.85,14.11,44.93,45.34,-0.41,53.97,0.00,0.01,0.13,0.44,0.00
8,Brighton,38,13.84,11.75,12.40,51.50,46.48,5.01,53.28,0.00,0.01,0.06,0.36,0.00
9,Fulham,38,15.25,6.66,16.09,52.09,55.16,-3.07,52.42,0.00,0.00,0.04,0.22,0.00


The final table shows the predicted league table with the probabilities of each team finishing 1st, Top 4, Top 6, Top 8 and Bottom 3

### LIMITATIONS AND FURTHER IMPROVEMENTS ###

- The ratings do not take into account short term fluctuations in form due to player transfers, injuries, suspensions and manager changes.
- Historic betting lines are usually the best predictor of future results, however they are a function of previous xG and goals performance. Combining these factors to create ratings runs the risk of them being cross correlated.
- The model does not currently have a way of assessing the performance of newly promoted teams. A more advanced model could take their performance in the lower division and apply a promotion factor that re-evaluates their performance in context of the higher division. After a few matches cross referencing betting lines against other teams can give a picture of how the market rates the newly promoted teams compared to their opponents.
- The rolling averages used to assess teams historic performance could include a weighting factor that weights more recent preformances more than older performances. This approach runs the risk of skewing team performance ratings if they have a particularly tough run of fixtures, however that could be mitigated by assessing teams performance in the context of their opponents strength.
- Assessing the model based on its mae against historic betting lines is just one way of measuring performance. Other techniques could include looking at the root mean squared error (rmse) if we wanted to prioritise minimising big differences, or mean absolute percentage error (MAPE) if we were worried about errors with high goal and supremacy lines skewing the results.
- Another way to make the model more robust would be to split the dataset into smaller subsets and train the model multiple times, each time using a different subset as the validation set. Checking the model has similar results across these subsets would protect against overfitting the model.
- The predictions will be used to find value in betting markets, so another way to assess its performance is to find historic examples of when the model would have identified value in the market and assess whether betting on these would have been profitable.